# 01. PydanticAI 에이전트 기본 구성

## 실험 목표
- PydanticAI + Google Gemini API 연결 확인
- 기본 에이전트 생성 및 데이터 관련 질문 응답 테스트
- System Prompt 설계 패턴 이해

## 기술 스택
- `pydantic-ai-slim[google]` — LLM 에이전트 프레임워크
- `google-generativeai` — Gemini API 백엔드
- `python-dotenv` — API 키 환경변수 관리

---
## 0. 환경 준비

In [ ]:
# 필요 패키지 설치 (최초 1회)
# !uv pip install pydantic-ai-slim[google] python-dotenv pandas

In [1]:
import os
from dotenv import load_dotenv
from pydantic_ai import Agent

# .env 파일에서 API 키 및 모델 로드
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL", 'gemini-3.1-flash-lite-preview')

# 모델 ID (pydantic-ai 문자열 방식)
model_id = f"google-gla:{gemini_model}"

# API 키 유효성 검사
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY가 설정되어 있지 않습니다.")

print(f"✅ API 키 로드 완료")
print(f"사용 모델: {model_id}")


✅ API 키 로드 완료
사용 모델: google-gla:gemini-3.1-flash-lite-preview


---
## 1. 기본 에이전트 생성

In [2]:
# DA 전문 에이전트 생성
da_agent = Agent(
    model=model_id,
    system_prompt="""
    당신은 데이터 분석 전문가입니다.
    사용자의 데이터 관련 질문에 간결하고 정확하게 한국어로 답변하십시오.
    통계적 근거와 함께 실용적인 인사이트를 제공하십시오.
    """
)

print("✅ DA 에이전트 생성 완료")

✅ DA 에이전트 생성 완료


In [4]:
print(f"사용 모델: {model_id}")

사용 모델: google-gla:gemini-3.1-flash-lite-preview


---
## 2. 기본 응답 테스트

In [7]:
import asyncio

# 간단한 데이터 분석 질문 테스트
async def test_basic_question():
    question = "평균, 중앙값, 최빈값의 차이를 설명하고 언제 각각을 사용해야 하는지 알려주세요."
    result = await da_agent.run(question)
    return result.output

response = await test_basic_question()
print("[질문] 평균, 중앙값, 최빈값의 차이")
print("-" * 60)
print(response)


[질문] 평균, 중앙값, 최빈값의 차이
------------------------------------------------------------
데이터 분석에서 **평균(Mean), 중앙값(Median), 최빈값(Mode)**은 데이터의 중심 경향성을 나타내는 지표이지만, 데이터의 분포에 따라 그 활용도가 달라집니다.

---

### 1. 정의 및 차이점

*   **평균(Mean):** 모든 값을 더해 데이터 개수로 나눈 값입니다. 전체 데이터의 총합 정보를 담고 있습니다.
*   **중앙값(Median):** 데이터를 크기순으로 나열했을 때 가장 중앙에 위치한 값입니다. 데이터의 양적 중간 지점을 나타냅니다.
*   **최빈값(Mode):** 데이터에서 가장 자주 등장하는 값입니다. 데이터의 인기나 빈도 특성을 나타냅니다.

---

### 2. 언제 무엇을 사용해야 하는가?

#### **평균을 사용할 때: "데이터가 정규분포를 이룰 때"**
*   **특징:** 모든 데이터 값을 반영하므로 수학적 처리에 유리합니다.
*   **주의점:** **이상치(Outlier)**에 매우 취약합니다. 예를 들어, 10명의 연봉 평균을 낼 때 억만장자 1명이 포함되면 평균값이 실제보다 훨씬 높게 왜곡됩니다.
*   **사례:** 성적 평균, 평균 기온 등 일반적인 분포.

#### **중앙값을 사용할 때: "데이터에 이상치가 있거나 비대칭일 때"**
*   **특징:** 이상치의 영향을 거의 받지 않으며, 전체의 '보통' 수준을 가장 잘 반영합니다.
*   **사례:** **주택 가격, 개인 소득, 자산 규모** 등 한두 명의 거액이 평균을 왜곡할 수 있는 경우 중앙값이 실제 체감 물가나 소득 수준을 더 정확하게 대변합니다.

#### **최빈값을 사용할 때: "범주형 데이터이거나 유행을 파악할 때"**
*   **특징:** 숫자가 아닌 데이터(이름, 지역, 선호도 등)에도 사용할 수 있습니다.
*   **사례:** 의류 사이즈, 가장 많이 팔린 제품, 선호하는

In [8]:
# 데이터 분석 워크플로우 관련 질문
async def test_workflow_question():
    question = """
    CSV 파일을 처음 받았을 때 데이터 분석가가 수행해야 할
    EDA(탐색적 데이터 분석) 단계를 순서대로 정리해주세요.
    """
    result = await da_agent.run(question)
    return result.output

response2 = await test_workflow_question()
print("[질문] EDA 워크플로우")
print("-" * 60)
print(response2)

[질문] EDA 워크플로우
------------------------------------------------------------
데이터 분석가로서 새로운 CSV 파일을 받았을 때 수행하는 **EDA(Exploratory Data Analysis) 6단계**를 정리해 드립니다.

### 1. 데이터 구조 및 메타데이터 확인 (Inspection)
데이터의 크기와 형태를 파악하여 전체적인 규모를 가늠합니다.
*   **체크리스트:** 행/열 개수(`shape`), 데이터 타입(`dtypes`), 컬럼명 확인.
*   **목적:** 대규모 데이터인지, 데이터 타입이 적절하게 설정되었는지(예: 날짜가 객체로 되어있지는 않은지) 파악합니다.

### 2. 결측치 및 중복값 처리 (Data Cleaning)
데이터의 품질을 결정짓는 가장 중요한 단계입니다.
*   **체크리스트:** `isnull().sum()`으로 결측치 비중 확인, `duplicated()`로 중복 데이터 제거.
*   **통계적 근거:** 결측치가 5% 미만이면 제거, 30% 이상이면 해당 컬럼의 유용성을 재검토합니다. 단순히 평균으로 채우는 것은 편향(bias)을 유발할 수 있으므로 중앙값이나 모델 기반 보간법을 고려해야 합니다.

### 3. 기술 통계량 분석 (Univariate Analysis)
개별 변수의 분포를 파악하여 데이터의 특이점(Outlier)을 찾습니다.
*   **체크리스트:** 평균, 중앙값, 표준편차, 사분위수(IQR) 확인.
*   **인사이트:** 평균과 중앙값의 차이가 크다면 데이터가 비대칭(skewed)된 상태이므로, 이상치 여부를 `boxplot`으로 반드시 확인해야 합니다.

### 4. 변수 간 상관관계 분석 (Multivariate Analysis)
변수들이 서로 어떤 영향을 미치는지 파악합니다.
*   **체크리스트:** 상관계수 히트맵(`heatmap`), 산점도(`scatter plot`).
*   **통계적 근거:** 피어슨 상관계수(r)가 0.

---
## 3. 대화 히스토리 활용 (멀티턴)

In [9]:
# 멀티턴 대화 테스트
async def test_multiturn():
    # 첫 번째 질문
    result1 = await da_agent.run("결측값 처리 방법에는 어떤 것들이 있나요?")
    print("[1차 질문] 결측값 처리 방법")
    print(result1.output)
    print()
    
    # 이전 대화 히스토리를 유지하면서 후속 질문
    result2 = await da_agent.run(
        "그 중에서 수치형 컬럼에 중앙값을 사용하는 게 좋은 경우는 언제인가요?",
        message_history=result1.new_messages()
    )
    print("[2차 질문] 중앙값 대체가 좋은 경우")
    print(result2.output)

await test_multiturn()

[1차 질문] 결측값 처리 방법
데이터 분석에서 결측값(Missing Value)을 처리하는 방법은 데이터의 성격과 결측의 이유(MCAR, MAR, MNAR)에 따라 달라집니다. 주요 방법 4가지를 정리해 드립니다.

### 1. 삭제법 (Deletion)
*   **방법:** 결측값이 포함된 행(Listwise) 또는 열(Column)을 제거합니다.
*   **장점:** 구현이 매우 간단하고 분석 결과의 왜곡이 적습니다.
*   **단점:** 데이터 손실이 크며, 결측이 무작위가 아닐 경우 분석 편향(Bias)이 발생합니다.
*   **추천:** 결측 비율이 매우 적고(보통 5% 미만), 데이터가 충분할 때 사용합니다.

### 2. 단순 대치법 (Simple Imputation)
*   **방법:** 결측값을 통계적 대표값으로 채웁니다. (평균, 중앙값, 최빈값 등)
*   **장점:** 데이터 손실 없이 모든 데이터를 활용할 수 있습니다.
*   **단점:** 변수 간의 상관관계가 왜곡되고, 표준오차가 과소평가되어 통계적 유의성이 과장될 수 있습니다.
*   **추천:** 수치형 데이터에는 **중앙값(Median)**을 사용하는 것이 이상치(Outlier) 영향이 적어 권장됩니다.

### 3. 다중 대치법 (Multiple Imputation)
*   **방법:** 결측값을 여러 번 예측하여 여러 개의 완성된 데이터셋을 만든 뒤, 분석 결과를 통합합니다. (MICE 알고리즘 등)
*   **장점:** 불확실성을 통계적으로 반영하여 매우 정교한 결과를 제공합니다.
*   **단점:** 계산 복잡도가 높고 해석이 어렵습니다.
*   **추천:** 결측이 특정 규칙을 가질 때(MAR) 가장 신뢰할 수 있는 방법입니다.

### 4. 예측 모델 기반 대치 (Model-based Imputation)
*   **방법:** 회귀 분석, KNN(K-Nearest Neighbors), 랜덤 포레스트 등을 이용해 결측값을 예측하여 채웁니다.
*   **장점:** 변수 

---
## 4. 에이전트 응답 메타데이터 확인

In [10]:
# 응답 메타데이터 (토큰 사용량 등) 확인
async def check_metadata():
    result = await da_agent.run("이상치(Outlier)란 무엇인가요? 한 문장으로 설명해주세요.")
    
    print("[응답 내용]")
    print(result.output)
    print()
    print("[응답 메타데이터]")
    print(f"- 응답 타입: {type(result.output).__name__}")
    print(f"- 사용 모델: {result.usage()}")

await check_metadata()

[응답 내용]
이상치란 전체 데이터의 일반적인 패턴에서 크게 벗어나 다른 관측값들과는 확연히 다른 특성을 보이는 데이터 포인트를 의미합니다.

[응답 메타데이터]
- 응답 타입: str
- 사용 모델: RunUsage(input_tokens=68, output_tokens=33, details={'text_prompt_tokens': 68}, requests=1)


---
## 5. 정리 및 다음 단계

### 학습 요약
| 항목 | 내용 |
|------|------|
| 모델 | `gemini-3.1-flash-lite-preview` |
| 에이전트 생성 | `Agent(model, system_prompt)` |
| 응답 호출 | `await agent.run(prompt)` |
| 멀티턴 | `message_history=result.new_messages()` |

### 다음 노트북 (02)
실제 데이터 분석에 활용할 **Tool 함수들을 정의**하고
에이전트에 등록하여 pandas 기반 분석 작업을 수행하도록 한다.